# Structured Streaming — 02: File Source Streaming

The file source watches a directory for new files and processes them as a stream.

- Key options: `maxFilesPerTrigger`, `latestFirst`, `cleanSource`
- Use cases: log ingestion, daily drops, sensor CSV uploads
- This notebook: writes CSV files to a watched dir in a background thread while the stream reads them

## Setup

In [ ]:
import tempfile, os, time, threading, csv, random, shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, input_file_name, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (
    SparkSession.builder
    .appName("SS-02-FileSource")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 1. Directory Setup — Watched and Output Paths

The file source requires a fixed schema (cannot infer at runtime since no file may exist yet).
We create two directories: `INPUT_DIR` (where new CSV files land) and `CHECKPOINT_DIR`.

In [ ]:
BASE = tempfile.mkdtemp().replace('\\', '/') + '/ss02'
INPUT_DIR     = BASE + '/input'
OUTPUT_DIR    = BASE + '/output'
CHECKPOINT    = BASE + '/checkpoint'

for d in [INPUT_DIR, OUTPUT_DIR, CHECKPOINT]:
    os.makedirs(d, exist_ok=True)

print("INPUT_DIR   :", INPUT_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("CHECKPOINT  :", CHECKPOINT)

## 2. Define the Schema

The file source cannot infer schema (the directory may be empty at start).
Always provide an explicit schema.

In [ ]:
# Schema matches the CSV files we will generate
order_schema = StructType([
    StructField("order_id",    IntegerType(), True),
    StructField("customer",    StringType(),  True),
    StructField("product",     StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("event_time",  StringType(),  True),   # kept as string for simplicity
])
print("Schema defined:")
spark.createDataFrame([], order_schema).printSchema()

## 3. Background File Writer

We simulate an external system dropping CSV files into `INPUT_DIR` every 4 seconds.
The streaming query picks them up automatically.

In [ ]:
CUSTOMERS = ["Alice", "Bob", "Carol", "Dave", "Eve"]
PRODUCTS  = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headset"]

def write_batch(batch_num):
    path = os.path.join(INPUT_DIR, f"orders_batch_{batch_num:03d}.csv")
    rows = [
        [
            batch_num * 10 + i,
            random.choice(CUSTOMERS),
            random.choice(PRODUCTS),
            random.randint(1, 5),
            round(random.uniform(10.0, 500.0), 2),
            time.strftime("%Y-%m-%d %H:%M:%S")
        ]
        for i in range(5)
    ]
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["order_id", "customer", "product", "quantity", "unit_price", "event_time"])
        writer.writerows(rows)
    print(f"  [Writer] Wrote batch {batch_num} -> {os.path.basename(path)}")

def file_writer_thread(n_batches=5, interval=4):
    for i in range(1, n_batches + 1):
        time.sleep(interval)
        write_batch(i)
    print("  [Writer] All batches written.")

print("File writer function ready.")

## 4. Start the Streaming Query

`maxFilesPerTrigger=1` means Spark processes one new file per micro-batch —
ideal for ordered, controlled ingestion.

In [ ]:
stream_df = (
    spark.readStream
    .format("csv")
    .schema(order_schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", "1")
    .load(INPUT_DIR)
    .withColumn("source_file", input_file_name())
    .withColumn("ingested_at", current_timestamp())
)

print("isStreaming:", stream_df.isStreaming)
stream_df.printSchema()

## 5. Write to Memory Sink and Start Background Writer

In [ ]:
# Start streaming query writing to memory sink
query = (
    stream_df
    .writeStream
    .format("memory")
    .queryName("orders_stream")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .start()
)

# Start background thread to drop CSV files
writer_thread = threading.Thread(target=file_writer_thread, args=(5, 4), daemon=True)
writer_thread.start()

print("Streaming query started. Waiting for 26 seconds...")
time.sleep(26)
writer_thread.join(timeout=5)
print("Done waiting.")

## 6. Inspect Results

In [ ]:
result = spark.sql("SELECT * FROM orders_stream ORDER BY order_id")
print(f"Total rows ingested: {result.count()}")
result.show(20, truncate=False)

# Per-file stats
print("\nRows per source file:")
spark.sql("""
    SELECT source_file, COUNT(*) as row_count
    FROM orders_stream
    GROUP BY source_file
    ORDER BY source_file
""").show(truncate=False)

## 7. Query Progress — How Many Files Were Processed?

In [ ]:
import json
progress = query.lastProgress
if progress:
    print("Last progress report:")
    print(json.dumps(progress, indent=2, default=str))
else:
    print("No progress recorded yet.")
    print("Recent progress count:", len(query.recentProgress))

query.stop()
print("Query stopped.")

## 8. File Source Options Reference

| Option | Default | Effect |
|---|---|---|
| `maxFilesPerTrigger` | all | Max new files per micro-batch |
| `latestFirst` | false | Process newest files first |
| `cleanSource` | off | `delete`/`archive` source files after processing |
| `sourceArchiveDir` | — | Where to move files when `cleanSource=archive` |
| `fileNameOnly` | false | Track by filename only, not full path |

## 9. Parquet File Source (Schema-on-Read)

Parquet files carry their schema — no need to specify one for parquet.
Demonstrate writing some Parquet then streaming from it.

In [ ]:
PARQUET_INPUT  = BASE + '/parquet_input'
PARQUET_CKPT   = BASE + '/parquet_ckpt'
os.makedirs(PARQUET_INPUT, exist_ok=True)

# Write a parquet file to the watched directory
import pyspark.sql.functions as F
sample = spark.range(50).withColumn("value", F.rand(seed=1) * 100)
sample.write.mode("overwrite").parquet(PARQUET_INPUT + "/batch_001.parquet")
print("Wrote initial parquet file.")

parquet_stream = (
    spark.readStream
    .format("parquet")
    .option("maxFilesPerTrigger", "1")
    .load(PARQUET_INPUT)
)

print("isStreaming:", parquet_stream.isStreaming)
parquet_stream.printSchema()

pq_query = (
    parquet_stream
    .writeStream
    .format("memory")
    .queryName("parquet_stream")
    .outputMode("append")
    .option("checkpointLocation", PARQUET_CKPT)
    .start()
)

time.sleep(6)
print("Parquet stream rows:", spark.sql("SELECT COUNT(*) AS n FROM parquet_stream").first()["n"])
pq_query.stop()

## Key Takeaways

- File source watches a directory — any new file matching the path glob is a micro-batch
- Always provide an explicit schema; do not rely on inference
- `maxFilesPerTrigger=1` gives ordered, predictable ingestion
- `input_file_name()` adds provenance — which file a row came from
- Parquet/ORC/JSON sources work the same way; schema required for JSON and CSV

## Practice Exercises

1. Change `maxFilesPerTrigger` to 2 — do you get fewer micro-batches?
2. Add a `filter(col("quantity") > 2)` before writing — confirm only high-qty orders land in memory.
3. Replace the CSV source with JSON (write JSON files in the writer thread, adjust schema).
4. Use `cleanSource="archive"` with `sourceArchiveDir` and confirm files move after processing.

## Teardown

In [ ]:
shutil.rmtree(BASE, ignore_errors=True)
print("Temp directories cleaned up.")
spark.stop()
print("SparkSession stopped.")